# EDA and Light Modeling

Network Experience, OLT Congestion & Customer Risk Analytics.

## Project Overview

This notebook uses only anonymized clean data. The churn-risk label is rule-based and should not be interpreted as a production churn model.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

DATA_DIR = Path('..') / 'data' / 'clean'
plt.style.use('seaborn-v0_8-whitegrid')

## Load Clean Data

In [ ]:
customers = pd.read_csv(DATA_DIR / 'customers_clean.csv')
plans = pd.read_csv(DATA_DIR / 'plans_clean.csv')
usage = pd.read_csv(DATA_DIR / 'usage_logs_clean.csv')
complaints = pd.read_csv(DATA_DIR / 'complaints.csv')
customer_metrics = pd.read_csv(DATA_DIR / 'customer_daily_metrics.csv')
olt_metrics = pd.read_csv(DATA_DIR / 'olt_daily_metrics.csv')
olt_info = pd.read_csv(DATA_DIR / 'olt_info_clean.csv')

{ name: len(df) for name, df in {'customers': customers, 'plans': plans, 'usage': usage, 'complaints': complaints, 'customer_metrics': customer_metrics, 'olt_metrics': olt_metrics}.items() }

## Data Validation

In [ ]:
validation = pd.DataFrame([
    ['customers unique customer_id', customers['customer_id'].is_unique],
    ['usage unique customer_id + log_date', ~usage.duplicated(['customer_id','log_date']).any()],
    ['plans unique plan_id', plans['plan_id'].is_unique],
    ['experience score between 0 and 100', customer_metrics['experience_score'].between(0,100).all()],
    ['risk score between 0 and 100', customer_metrics['churn_risk_score'].between(0,100).all()],
], columns=['check','passed'])
validation

## Usage Analysis

Distribution of daily customer data usage.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
customer_metrics['data_usage_gb'].hist(bins=30, ax=ax, color='#3B82F6')
ax.set_title('Distribution of Daily Data Usage')
ax.set_xlabel('Data usage (GB)')
ax.set_ylabel('Customer-days')
plt.show()

## Latency Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
customer_metrics['latency_ms'].hist(bins=30, ax=ax, color='#F97316')
ax.set_title('Distribution of Latency')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Customer-days')
plt.show()

## Downtime Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
customer_metrics['downtime_minutes'].hist(bins=30, ax=ax, color='#EF4444')
ax.set_title('Distribution of Downtime')
ax.set_xlabel('Downtime minutes')
ax.set_ylabel('Customer-days')
plt.show()

## Daily Total Usage Trend Across the Network

In [ ]:
daily_usage = customer_metrics.groupby('log_date')['data_usage_gb'].sum()
fig, ax = plt.subplots(figsize=(8,4))
daily_usage.plot(ax=ax, marker='o', color='#2563EB')
ax.set_title('Daily Total Usage Trend')
ax.set_xlabel('Date')
ax.set_ylabel('Total usage (GB)')
plt.xticks(rotation=45)
plt.show()

## Customer Experience Analysis

In [ ]:
exp_by_segment = customer_metrics.groupby('value_segment')['experience_score'].mean().sort_values()
fig, ax = plt.subplots(figsize=(8,4))
exp_by_segment.plot(kind='barh', ax=ax, color='#10B981')
ax.set_title('Average Experience Score by Value Segment')
ax.set_xlabel('Average experience score')
plt.show()

## Churn Risk Analysis

In [ ]:
risk_counts = customer_metrics[customer_metrics['log_date'] == customer_metrics['log_date'].max()]['churn_risk_category'].value_counts()
fig, ax = plt.subplots(figsize=(7,4))
risk_counts.reindex(['Low Risk','Medium Risk','High Risk']).plot(kind='bar', ax=ax, color=['#10B981','#F59E0B','#EF4444'])
ax.set_title('Latest Churn Risk Category Distribution')
ax.set_xlabel('Risk category')
ax.set_ylabel('Customers')
plt.xticks(rotation=0)
plt.show()

## OLT Congestion Analysis

In [ ]:
olt_rank = olt_metrics.groupby('olt_id')['congestion_ratio_percent'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9,4))
olt_rank.plot(kind='bar', ax=ax, color='#6366F1')
ax.set_title('OLT Congestion Ranking')
ax.set_xlabel('OLT ID')
ax.set_ylabel('Average congestion ratio (%)')
plt.show()

## Complaint Leakage Analysis

In [ ]:
leakage = customer_metrics[customer_metrics['complaint_leakage_flag'] == 1]
leakage.groupby('value_segment')['customer_id'].nunique().sort_values(ascending=False).to_frame('leakage_customers')

## Correlation Matrix

In [ ]:
corr_cols = ['data_usage_gb','latency_ms','downtime_minutes','experience_score','usage_drop_percent','complaint_count','churn_risk_score']
corr = customer_metrics[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8,6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)), corr_cols, rotation=45, ha='right')
ax.set_yticks(range(len(corr_cols)), corr_cols)
fig.colorbar(im, ax=ax)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()
corr.round(2)

## Optional Light Modeling

This decision tree classifies the rule-based risk category to understand which factors contribute most to that proxy label. It is not a production churn prediction model.

In [ ]:
model_df = customer_metrics.dropna(subset=['usage_drop_percent']).copy()
features = ['usage_drop_percent','downtime_minutes','latency_ms','complaint_count','experience_score']
X = model_df[features]
y = model_df['churn_risk_category']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)
print(classification_report(y_test, preds))
importance = pd.Series(clf.feature_importances_, index=features).sort_values()
fig, ax = plt.subplots(figsize=(8,4))
importance.plot(kind='barh', ax=ax, color='#0EA5E9')
ax.set_title('Decision Tree Feature Importance')
plt.show()

## Final Insights

- Use congestion rankings to prioritize capacity review.
- Use complaint leakage to identify silent poor-experience customers.
- Treat churn risk as an explainable operating proxy, not a trained churn outcome model.
- Extend the dataset history before using trend claims in a production setting.